In [1]:
from transformers import AutoModelForCausalLM,AutoTokenizer
model = AutoModelForCausalLM.from_pretrained("model/Qwen3-0.6B")
tokenizer = AutoTokenizer.from_pretrained("model/Qwen3-0.6B")

c:\PycharmProjects\projects\fine_tune_proj\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipping import of cpp extensions due to incompatible torch version 2.12.1+cu130 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
Loading weights: 100%|██████████| 311/311 [00:00<00:00, 1317.16it/s, Materializing param=model.norm.weight]                              
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [2]:
#1、第一步：对数据集进行处理，处理成trl所需要的language modeling类型的对话格式
from datasets import load_dataset
# 1.1、将两个数据集文件，分别加载到dataset dict当中的train 和test分片中
dataset_dict = load_dataset("json",data_files = {"train":"data/keywords_data_train.jsonl","test":"data/keywords_data_test.jsonl"})
dataset_dict

DatasetDict({
    train: Dataset({
        features: ['conversation_id', 'category', 'conversation', 'dataset'],
        num_rows: 49500
    })
    test: Dataset({
        features: ['conversation_id', 'category', 'conversation', 'dataset'],
        num_rows: 500
    })
})

In [3]:
from typing import List,Dict
# 1.2 、定义数据处理的 mapping function
def convert_function(examples:Dict[str, List]):
    """
    数据集分批次调用该方法，修改数据格式
    """
    conversation_list:List[List[Dict]] = examples["conversation"]
    message_list = []
    for data in conversation_list:
        # data是单条样本
        coversation_dict = data[0]
        current_data_message_list = [
            {"role":"user","content":coversation_dict["human"]},
            {"role":"assistant","content":coversation_dict["assistant"]}
        ]
        message_list.append(current_data_message_list)
    
    return {"messages":message_list}

# 1.3、调用dataset_dict.map方法，传入mapping function，对数据进行格式化
mapped_dataset_dict = dataset_dict.map(convert_function,batched=True,remove_columns = ['conversation_id', 'category', 'conversation', 'dataset'])
mapped_dataset_dict

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 49500
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 500
    })
})

In [5]:
def get_data_length(examples:Dict[str, List]):
    """
    数据集分批次调用该方法，修改数据格式
    """
    data_messages_lists = examples["messages"]
    length_list = []
    for data in data_messages_lists:
        result = tokenizer.apply_chat_template(data,tokenize=True)
        length_list.append(len(result["input_ids"]))
    
    return {"length":length_list}



length_data = mapped_dataset_dict.map(get_data_length,batched=True,remove_columns=["messages"])

Map: 100%|██████████| 500/500 [00:00<00:00, 2858.28 examples/s]


In [8]:
df = length_data["train"].to_pandas()
df

,length
0,139
1,530
2,118
3,173
4,108
...,...
49495,238
49496,184
49497,325
49498,143


In [12]:
df["length"].quantile(0.999)

np.float64(654.5029999999897)

In [4]:
mapped_dataset_dict["train"][0]

{'messages': [{'content': '高氟铍矿石在熔炼过程中配入氢氧化铝来脱除其中的氟.结果表明,在配入5％Na2CO3、9.3％Al(OH)3、1400～1500℃熔炼20 min的情况下,BeO回收率达到96％以上,脱氟效果良好(铍玻璃F/BeO能控制在15％以内).为高氟铍矿石的工业应用探索出新的冶炼途径.\n找出上文中的关键词',
   'role': 'user'},
  {'content': '高氟铍矿;配料;熔炼;回收率;脱氟率', 'role': 'assistant'}]}

In [14]:
# 2、构造SFTTrainer实例
from trl.trainer.sft_trainer import SFTTrainer
from trl.trainer.sft_config import SFTConfig
# 2.1、构造训练时的参数SFTConfig
import os
os.environ["TENSORBOARD_LOGGING_DIR"] = "logs/05_trl_demo"
sft_config = SFTConfig(
    # 数据规模相关
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,
    max_steps=500,
    num_train_epochs=1,
    # 训练日志相关
    logging_strategy="steps",
    logging_steps=50,
    report_to="tensorboard",
    # 学习率和优化器相关
    learning_rate=3e-5,
    lr_scheduler_type="cosine",
    warmup_steps=0.1,
    # optim=
    # 评估和保存相关
    eval_strategy="steps",
    eval_steps=50,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    load_best_model_at_end=True,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    output_dir="finetuned/05_trl_demo",
    # 优化相关
    bf16=True,
    gradient_checkpointing=True,
    activation_offloading=True,
    max_length=700,

    # 其他参数：是否只对assistant answer计算损失、chat_template_path
    assistant_only_loss=True,
    chat_template_path= "./chat_template.jinja"
    
)

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = mapped_dataset_dict["train"],
    eval_dataset = mapped_dataset_dict["test"],
    args = sft_config
)

Truncating eval dataset: 100%|██████████| 500/500 [00:00<00:00, 33801.04 examples/s]


In [17]:
dataloader = trainer.get_train_dataloader()
for batch in dataloader:
    print(batch)
    decoded_result = tokenizer.decode(batch["input_ids"])
    print(decoded_result[0])
    break

{'input_ids': tensor([[151644,   8948,    198,  ..., 151643, 151643, 151643],
        [151644,   8948,    198,  ..., 151643, 151643, 151643],
        [151644,   8948,    198,  ..., 151643, 151643, 151643],
        [151644,   8948,    198,  ...,     29, 151645,    198]],
       device='cuda:0'), 'labels': tensor([[  -100,   -100,   -100,  ...,   -100,   -100,   -100],
        [  -100,   -100,   -100,  ...,   -100,   -100,   -100],
        [  -100,   -100,   -100,  ...,   -100,   -100,   -100],
        [  -100,   -100,   -100,  ...,     29, 151645,    198]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1]], device='cuda:0')}
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 13 July 2026
Reasoning Mode: /think

## Custom Instructions

You are a helpful AI assistant named SmolLM, trained by Hugging Face. Your role as an assistant invol

In [ ]:
trainer.train()
trainer.save_model("finetuned/05_trl_demo")